# Random Forest

**Problem with one tree:** a deep tree overfits. A small change in the training data gives a very different tree.

**Idea:** train many different trees and let them **vote**.

How the trees are made different:
1. **Bootstrap sample:** each tree gets its own training set. We draw rows at random **with replacement** (some rows twice, some not at all).
2. **Random features:** at each question, a tree only looks at a few random features (`max_features`, default √(number of features)).

The mistakes of different trees partly cancel out, so the forest is better than one tree.

What we test:
* **number of trees**
* **max_features** – how many features each question can look at
* **max_depth**

The tree code is the same as in `decision_tree/decision_tree.ipynb`.

## 1. Setup

In [ ]:
import os
import sys

# The notebook is in a sub-folder. Go to the project folder (where common.py is).
if not os.path.exists("common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import common   # our shared file: load data, split, metrics, Kaggle file

common.set_seed(42)
os.makedirs("random_forest/outputs", exist_ok=True)

## The tree code

**Impurity** – how mixed the classes are in a group of rows (0 = only one class):
* Gini: `1 - sum(p_c^2)`
* Entropy: `-sum(p_c * log2(p_c))`

`p_c` = share of class `c` in the group.

**Building the tree** (function `build`):
1. If we must stop (max depth reached, too few rows, or only one class left) → make a **leaf**. The leaf predicts the most common class.
2. Otherwise try many questions *"feature f <= threshold t ?"*. For each one, split the rows into left (yes) and right (no).
3. **Gain** = impurity before − weighted impurity after. Keep the question with the biggest gain.
4. Build the left child and the right child the same way (recursion).

**Faster search:** we do not try every value as a threshold.
* For every feature we only try 31 thresholds (its quantiles).
* At the start, every value is replaced by its **bin number** (which interval between thresholds it falls in).
* At each node, one `np.bincount` makes a table: for every feature and every bin, how many rows of each class.
* A cumulative sum over the bins gives the class counts on the left side for every threshold.
* So we get the gain of every (feature, threshold) question at once, and keep the biggest.

**Prediction:** start at the root, answer the question, go left or right, until a leaf.

In [ ]:
def gini(counts):
    """counts[..., c] = number of rows of class c in a group (last axis = classes).
    Returns one impurity per group."""
    totals = counts.sum(axis=-1, keepdims=True)
    p = counts / np.maximum(totals, 1)
    return 1 - (p ** 2).sum(axis=-1)


def entropy(counts):
    totals = counts.sum(axis=-1, keepdims=True)
    p = counts / np.maximum(totals, 1)
    log_p = np.log2(np.where(p > 0, p, 1))      # log2(1) = 0, so empty classes add nothing
    return -(p * log_p).sum(axis=-1)


IMPURITY = {"gini": gini, "entropy": entropy}


def make_thresholds(X, n_thresholds):
    """For every feature: n_thresholds candidate thresholds (quantiles). Shape (n_features, n_thresholds)."""
    q = np.linspace(0, 1, n_thresholds + 2)[1:-1]
    return np.quantile(X, q, axis=0).T


def to_bins(X, thresholds):
    """Bin number of every value = how many thresholds are smaller than it.
    So: value <= thresholds[f, t]  is the same as  bin <= t."""
    bins = np.zeros(X.shape, dtype=int)
    for f in range(X.shape[1]):
        bins[:, f] = np.searchsorted(thresholds[f], X[:, f])
    return bins


class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1,
                 criterion="gini", max_features=None, n_thresholds=31, seed=42):
        self.max_depth = max_depth                  # None = no limit
        self.min_samples_split = min_samples_split  # a node needs at least this many rows to be split
        self.min_samples_leaf = min_samples_leaf    # each child must keep at least this many rows
        self.criterion = criterion                  # "gini" or "entropy"
        self.max_features = max_features            # None = try all features (Random Forest: a random few)
        self.n_thresholds = n_thresholds
        self.rng = np.random.default_rng(seed)

    # ---------------- training ----------------
    def fit(self, X, y, n_classes):
        self.n_classes = n_classes
        self.impurity = IMPURITY[self.criterion]
        self.thresholds = make_thresholds(X, self.n_thresholds)
        bins = to_bins(X, self.thresholds)
        self.root = self.build(bins, y, depth=0)
        return self

    def build(self, bins, y, depth):
        counts = np.bincount(y, minlength=self.n_classes)
        leaf = {"leaf": True, "label": int(np.argmax(counts))}     # most common class

        if len(y) < self.min_samples_split:
            return leaf
        if self.max_depth is not None and depth >= self.max_depth:
            return leaf
        if counts.max() == len(y):                                  # only one class left
            return leaf

        feature, t = self.best_split(bins, y, counts)
        if feature is None:                                         # no split makes it better
            return leaf

        go_left = bins[:, feature] <= t
        return {"leaf": False,
                "feature": feature,
                "threshold": self.thresholds[feature, t],
                "left": self.build(bins[go_left], y[go_left], depth + 1),
                "right": self.build(bins[~go_left], y[~go_left], depth + 1)}

    def best_split(self, bins, y, counts):
        n_rows, n_features = bins.shape
        n_bins = self.n_thresholds + 1
        C = self.n_classes
        parent = self.impurity(counts)                              # impurity before the split

        features = np.arange(n_features)
        if self.max_features is not None:                           # Random Forest: a few random features
            features = self.rng.choice(n_features, self.max_features, replace=False)
        F = len(features)

        # table[f, b, c] = number of rows of class c that fall in bin b of feature f.
        # To count everything with ONE np.bincount, every (feature, bin, class) gets its own number:
        #     cell = (f * n_bins + b) * C + c
        cell = (np.arange(F) * n_bins + bins[:, features]) * C + y[:, None]
        table = np.bincount(cell.ravel(), minlength=F * n_bins * C).reshape(F, n_bins, C)

        # Threshold t sends bins 0..t to the left -> cumulative sum over the bins
        left = np.cumsum(table, axis=1)[:, :-1, :]      # shape (F, n_thresholds, C)
        right = counts - left
        n_left = left.sum(axis=2)
        n_right = right.sum(axis=2)

        after = (n_left * self.impurity(left) + n_right * self.impurity(right)) / n_rows
        gain = parent - after                           # shape (F, n_thresholds): one gain per question
        too_small = (n_left < self.min_samples_leaf) | (n_right < self.min_samples_leaf)
        gain[too_small] = 0

        f, t = np.unravel_index(np.argmax(gain), gain.shape)
        if gain[f, t] <= 1e-12:                         # no question makes the groups purer
            return None, None
        return features[f], t

    # ---------------- prediction ----------------
    def predict_one(self, x):
        node = self.root
        while not node["leaf"]:
            if x[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        return node["label"]

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])


def count_nodes(node):
    if node["leaf"]:
        return 1
    return 1 + count_nodes(node["left"]) + count_nodes(node["right"])


def tree_depth(node):
    if node["leaf"]:
        return 0
    return 1 + max(tree_depth(node["left"]), tree_depth(node["right"]))

## 2. The forest

`predict` lets every tree vote. The class with the most votes wins.
`n_trees` in `predict` lets us use only the first trees, to see how the number of trees changes the accuracy without training again.

In [ ]:
class RandomForest:
    def __init__(self, n_trees=50, max_features="sqrt", max_depth=None, seed=42):
        self.n_trees = n_trees
        self.max_features = max_features      # "sqrt", "log2" or a number
        self.max_depth = max_depth
        self.seed = seed

    def fit(self, X, y, n_classes):
        self.n_classes = n_classes
        n_rows, n_features = X.shape
        if self.max_features == "sqrt":
            m = int(np.sqrt(n_features))
        elif self.max_features == "log2":
            m = int(np.log2(n_features))
        else:
            m = int(self.max_features)

        rng = np.random.default_rng(self.seed)
        self.trees = []
        for i in range(self.n_trees):
            rows = rng.integers(0, n_rows, n_rows)          # bootstrap: random rows, with replacement
            tree = DecisionTree(max_depth=self.max_depth, max_features=m, seed=self.seed + i)
            tree.fit(X[rows], y[rows], n_classes)
            self.trees.append(tree)
        return self

    def predict(self, X, n_trees=None):
        trees = self.trees if n_trees is None else self.trees[:n_trees]
        votes = np.zeros((len(X), self.n_classes))
        for tree in trees:
            pred = tree.predict(X)
            votes[np.arange(len(X)), pred] += 1             # one vote per tree
        return votes.argmax(axis=1)

## 3. Load the data

In [ ]:
X_image, y_image, X_image_test, image_test_ids = common.load_task("image")
X_sent, y_sent, X_sent_test, sent_test_ids = common.load_task("sentiment")

DATA = {"image": (X_image, y_image), "sentiment": (X_sent, y_sent)}
TEST = {"image": (X_image_test, image_test_ids), "sentiment": (X_sent_test, sent_test_ids)}
N_CLASSES = common.N_CLASSES    # {"image": 50, "sentiment": 3}


def prepare(task):
    """Same 80/20 split as every model. No preprocessing: a tree compares one feature
    with a threshold, so scaling a feature only moves the threshold."""
    X, y = DATA[task]
    train_idx, val_idx = common.stratified_split(y)
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]

## 4. Number of trees

We train one forest of 50 trees and check the accuracy with the first 1, 5, 10, 25 and 50 trees.

In [ ]:
results = []
forests = {}
TREE_COUNTS = [1, 5, 10, 25, 50]

for task in ["sentiment", "image"]:
    X_tr, y_tr, X_va, y_va = prepare(task)
    start = time.time()
    forest = RandomForest(n_trees=50).fit(X_tr, y_tr, N_CLASSES[task])
    forests[task] = forest
    print(f"{task}: 50 trees trained in {time.time() - start:.0f}s")

    check = np.random.default_rng(42).choice(len(X_tr), 1000, replace=False)   # 1000 train rows
    train_acc, val_acc = [], []
    for n in TREE_COUNTS:
        train_acc.append(common.accuracy(y_tr[check], forest.predict(X_tr[check], n)))
        val_acc.append(common.accuracy(y_va, forest.predict(X_va, n)))
        results.append({"task": task, "n_trees": n, "max_features": "sqrt", "max_depth": "None",
                        "val_acc": round(val_acc[-1], 4)})
        print(f"   {n:3d} trees: train {train_acc[-1]:.3f}   validation {val_acc[-1]:.3f}")

    plt.figure(figsize=(6, 3.5))
    plt.plot(TREE_COUNTS, train_acc, marker="o", label="train")
    plt.plot(TREE_COUNTS, val_acc, marker="o", label="validation")
    plt.xscale("log")
    plt.xticks(TREE_COUNTS, [str(n) for n in TREE_COUNTS])
    plt.xlabel("number of trees")
    plt.ylabel("accuracy")
    plt.title(f"{task}: number of trees")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

## 5. Features per question, and depth (20 trees each, to save time)

* Fewer features per question → trees are more different (good for voting), but each tree is weaker.
* A depth limit makes every tree simpler.

In [ ]:
def try_forest(task, **settings):
    X_tr, y_tr, X_va, y_va = prepare(task)
    start = time.time()
    forest = RandomForest(n_trees=20, **settings).fit(X_tr, y_tr, N_CLASSES[task])
    acc = common.accuracy(y_va, forest.predict(X_va))
    results.append({"task": task, "n_trees": 20,
                    "max_features": str(settings.get("max_features", "sqrt")),
                    "max_depth": str(settings.get("max_depth", None)),
                    "val_acc": round(acc, 4)})
    print(f"{task:<9} {str(settings):<30} validation {acc:.3f}   ({time.time() - start:.0f}s)")


for task in ["sentiment", "image"]:
    for mf in ["log2", "sqrt", 50]:
        try_forest(task, max_features=mf)
    for depth in [10, 20]:
        try_forest(task, max_depth=depth)

In [ ]:
table = pd.DataFrame(results)
for task in ["sentiment", "image"]:
    t = table[table["task"] == task].sort_values("val_acc", ascending=False)
    t.to_csv(f"random_forest/outputs/{task}_experiments.csv", index=False)
    display(t)

## 6. Final forest: error analysis

We use the 50-tree forest from section 4 (`max_features = "sqrt"`, no depth limit).
If section 5 finds clearly better settings, change `FINAL` below.

In [ ]:
def show_errors(task, y_true, y_pred):
    cm = common.confusion_matrix(y_true, y_pred, N_CLASSES[task])
    if task == "sentiment":
        print(pd.DataFrame(cm, index=common.SENTIMENT_NAMES, columns=common.SENTIMENT_NAMES))
        recall = common.per_class_recall(cm)
        for i, name in enumerate(common.SENTIMENT_NAMES):
            print(f"   {name:<9} recall {recall[i]:.3f}")
    else:
        print("   most common mistakes (true class, predicted class, count):")
        for true_c, pred_c, count in common.most_confused_pairs(cm, 5):
            print(f"      {true_c:2d} -> {pred_c:2d}   {count} times")

In [ ]:
FINAL = {"n_trees": 50, "max_features": "sqrt", "max_depth": None}

for task in ["sentiment", "image"]:
    X_tr, y_tr, X_va, y_va = prepare(task)
    if FINAL == {"n_trees": 50, "max_features": "sqrt", "max_depth": None}:
        forest = forests[task]                      # already trained in section 4
    else:
        forest = RandomForest(**FINAL).fit(X_tr, y_tr, N_CLASSES[task])
    pred = forest.predict(X_va)
    np.save(f"random_forest/outputs/{task}_val_pred.npy", pred)     # used by final/compare.ipynb
    print(f"\n{task}: validation accuracy {common.accuracy(y_va, pred):.4f}")
    show_errors(task, y_va, pred)

## 7. Test predictions for Kaggle

Same settings, trained on **all** training rows.

In [ ]:
for task in ["sentiment", "image"]:
    X, y = DATA[task]
    X_test, test_ids = TEST[task]
    start = time.time()
    forest = RandomForest(**FINAL).fit(X, y, N_CLASSES[task])
    pred = forest.predict(X_test)
    pd.DataFrame({"id": test_ids, "label": pred}).to_csv(f"random_forest/outputs/{task}_test_pred.csv", index=False)
    print(f"{task}: saved random_forest/outputs/{task}_test_pred.csv  ({time.time() - start:.0f}s)")

## 8. Notes for the presentation

* How much better is the forest than one Decision Tree? Why? (Voting of many different trees reduces overfitting.)
* Train accuracy stays high, but validation still goes up with more trees: more trees do not make the forest overfit more.
* Effect of `max_features`: strong trees vs different trees.
* Trees are still weaker than k-NN and the neural network on these features. Why?